![image.png](attachment:44d2f751-c0ff-48be-8e86-77b4f69628a6.png)

#### Passo 1: Instalar bibliotecas

In [ ]:
#!pip install requests
#!pip install beautifulsoup4
#!pip install sqlite3
#!pip install pandas

#### Passo 2: Importar bibliotecas

In [ ]:
import requests
from bs4 import BeautifulSoup
import sqlite3
import pandas as pd

#### Passo 3: Fromatar link de busca

Neste exemplo utilizaremos um site público de testes de scraping

http://books.toscrape.com (feito exatamente para treinar scraping).

In [ ]:
# URL base
url = "https://books.toscrape.com/catalogue/category/books_1/page-1.html"

#### Passo 4: Determinar variáveis que para armazenamento
Neste exemplo, iremos utilizar:
- Título
- Preços
- Estoque

In [ ]:
titulos, precos, estoque = [], [], []

#### Passo 5: Obter informações do link

In [ ]:
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

#### Passo 5.1: Obter informações dos produtos

In [ ]:
# Cada produto está dentro de article.product_pod
livros = soup.find_all("article", class_="product_pod")

In [ ]:
livros[0].h3.a['title']

'A Light in the Attic'

In [ ]:
# Preço
livros[0].find("p", class_="price_color").text.replace("Â£", "")

'51.77'

In [ ]:
# Estoque
livros[0].find("p", class_="instock availability").text.strip()
# A função strip()em Python serve para remover espaços em branco ou caracteres específicos do início e/ou fim de uma string

'In stock'

#### Passo 6: Coletar dados em escala
Neste exemplo, vamos coletar dados das 5 primeiras páginas

In [ ]:
for page in range(1, 5):
    response = requests.get(f"https://books.toscrape.com/catalogue/category/books_1/page-{page}.html")
    soup = BeautifulSoup(response.text, "html.parser")

    # Cada produto está dentro de article.product_pod
    livros = soup.find_all("article", class_="product_pod")

    for livro in livros:
        # Título
        titulo = livro.h3.a["title"]
        titulos.append(titulo)

        # Preço
        preco = livro.find("p", class_="price_color").text.replace("Â£", "")
        precos.append(preco)

        # Estoque
        estoque_info = livro.find("p", class_="instock availability").text.strip()
        estoque.append(estoque_info)

print("Coleta concluída!")

Coleta concluída!


#### Passo 7: Criar DataFrame

In [ ]:
df = pd.DataFrame({"titulo": titulos, "preco": precos, "estoque": estoque})
df.head()

,titulo,preco,estoque
0,A Light in the Attic,51.77,In stock
1,Tipping the Velvet,53.74,In stock
2,Soumission,50.10,In stock
3,Sharp Objects,47.82,In stock
4,Sapiens: A Brief History of Humankind,54.23,In stock


#### Passo 8: Criar Banco de Dados

In [ ]:
# Conectar ao banco (cria arquivo se não existir)
conn = sqlite3.connect("livros.db")
cursor = conn.cursor()

#### Passo 9: DDL – Data Definition Language
DDL serve para **definir a estrutura do banco de dados**: criar, alterar e excluir tabelas.

- `CREATE` → cria tabelas e estruturas  
- `ALTER` → modifica a estrutura de tabelas  
- `DROP` → apaga tabelas ou bancos de dados  


In [ ]:
# CREATE TABLE → Cria tabela
cursor.execute("""
CREATE TABLE IF NOT EXISTS livros (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    titulo TEXT NOT NULL,
    preco REAL NOT NULL,
    estoque TEXT NOT NULL
);
""")
print("Tabela livros criada")

Tabela livros criada


##### 9.1: Listar todas tabelas do banco
Neste exemplo iremos utilizaremos a sintaxe `SELECT name FROM sqlite_master;`

- `sqlite_master` é uma tabela especial **interna** do SQLite que guarda **metadados** sobre o banco de dados: tabelas, índices, views e triggers
- `SELECT name` seleciona apenas o nome (`name`) dos objetos armazenados no `sqlite_master`

In [ ]:
cursor.execute("SELECT name FROM sqlite_master;")
print(cursor.fetchall()) # Recupera todos os resultados da última consulta feita com cursor.execute(...)

[('livros',), ('sqlite_sequence',), ('vendas',)]


Como pegar os resultados usando o método `fetch` (buscar)
- `fetchone()` Retorna apenas uma linha do resultado (a próxima disponível). Se não houver mais nada, retorna `None`
- `fetchmany(n)` Retorna as próximas n linhas do resultado.
- `fetchall()` Retorna todas as linhas restantes do resultado em uma lista de tuplas.

##### 9.2: Verificar o esquema da tabela (estrutura das colunas)

In [ ]:
cursor.execute("PRAGMA table_info(livros);")
for row in cursor.fetchall():
    print(row)

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'titulo', 'TEXT', 1, None, 0)
(2, 'preco', 'REAL', 1, None, 0)
(3, 'estoque', 'TEXT', 1, None, 0)


A saída mostra cada coluna da tabela:

(`cid` , `name`  ,`type` , `notnull` , `dflt_value` , `pk`)

- `cid` índice da coluna
- `name` nome da coluna
- `type` tipo de dado (TEXT, REAL, etc.)
- `notnull` se pode ser NULL (0 = sim, 1 = não)
- `dflt_value` valor padrão (se houver)
- `pk` se é chave primária (1 = sim)

##### 9.3: Alterar tabela
Nesta etapa utilizaremos o comando `ALTER TABLE` informar a alteração da tabela e o comando `ADD COLUMN` para especificar a alteração

In [ ]:
cursor.execute("ALTER TABLE livros ADD COLUMN categoria TEXT;")
conn.commit()
print("Coluna categoria adicionada à tabela livros")

Coluna categoria adicionada à tabela livros


##### 9.4: Verifica o equema da tabela após incluir coluna coluna

In [ ]:
cursor.execute("PRAGMA table_info(livros);")
for row in cursor.fetchall():
    print(row)

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'titulo', 'TEXT', 1, None, 0)
(2, 'preco', 'REAL', 1, None, 0)
(3, 'estoque', 'TEXT', 1, None, 0)
(4, 'categoria', 'TEXT', 0, None, 0)


##### 9.5: Remover uma coluna
Nesta etapa utilizaremos o comando `ALTER TABLE` informar a alteração da tabela e o comando `DROP COLUMN` para especificar a alteração

In [ ]:
cursor.execute("ALTER TABLE livros DROP COLUMN categoria")
conn.commit()
print("Coluna removida com sucesso!")

Coluna removida com sucesso!


##### 9.6: Verifica o equema da tabela após incluir coluna coluna

In [ ]:
cursor.execute("PRAGMA table_info(livros);")
for row in cursor.fetchall():
    print(row)

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'titulo', 'TEXT', 1, None, 0)
(2, 'preco', 'REAL', 1, None, 0)
(3, 'estoque', 'TEXT', 1, None, 0)


##### 9.7: Excluir tabela
Primeiro vamos criar uma tabela adicional

In [ ]:
# CREATE TABLE → Cria tabela
cursor.execute("""
CREATE TABLE IF NOT EXISTS produtos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    titulo TEXT NOT NULL,
    preco REAL NOT NULL,
    estoque TEXT NOT NULL
);
""")
print("Tabela produtos criada")

Tabela produtos criada


##### Listar todas tabelas do banco

In [ ]:
cursor.execute("SELECT name FROM sqlite_master;")
print(cursor.fetchall()) # Recupera todos os resultados da última consulta feita com cursor.execute(...)

[('livros',), ('sqlite_sequence',), ('vendas',), ('produtos',)]


##### Remover tabela do banco

In [ ]:
cursor.execute("DROP TABLE IF EXISTS produtos")
conn.commit()
print("Tabela excluída com sucesso!")

Tabela excluída com sucesso!


##### Listar todas tabelas do banco

In [ ]:
cursor.execute("SELECT name FROM sqlite_master;")
print(cursor.fetchall()) # Recupera todos os resultados da última consulta feita com cursor.execute(...)

[('livros',), ('sqlite_sequence',), ('vendas',)]


#### Passo 10: DML – Data Manipulation Language

DML serve para **manipular os dados** dentro das tabelas:

- `INSERT` → inserir registros  
- `UPDATE` → atualizar registros  
- `DELETE` → excluir registros



##### 10.1 `INSERT` (inserir dados)

In [ ]:
for _, row in df.iterrows():
    cursor.execute("INSERT INTO livros (titulo, preco, estoque) VALUES (?, ?, ?)",
                   (row["titulo"], row["preco"], row["estoque"]))
conn.commit()
print("Dados armazenados no banco!")

Dados armazenados no banco!


##### Verifica quantos registros foram inseridos

In [ ]:
cursor.execute("SELECT COUNT(*) FROM livros;")
cursor.fetchone()

(167,)

##### Verifica uma amostra dos registros inseridos

In [ ]:
cursor.execute("SELECT * FROM livros;")
cursor.fetchmany(5)

[(1, 'A Light in the Attic', 49.9, 'In stock'),
 (2, 'Tipping the Velvet', 53.74, 'In stock'),
 (3, 'Soumission', 50.1, 'In stock'),
 (4, 'Sharp Objects', 47.82, 'In stock'),
 (5, 'Sapiens: A Brief History of Humankind', 54.23, 'In stock')]

##### 10.2: `UPDATE` (atualizar dados)

In [ ]:
cursor.execute("UPDATE livros SET preco = 49.90 WHERE titulo = 'A Light in the Attic';")
conn.commit()
print("Valor do livro atualizado")

Valor do livro atualizado


##### Verifica uma amostra dos registros inseridos

In [ ]:
cursor.execute("SELECT * FROM livros;")
cursor.fetchmany(5)

[(1, 'A Light in the Attic', 49.9, 'In stock'),
 (2, 'Tipping the Velvet', 53.74, 'In stock'),
 (3, 'Soumission', 50.1, 'In stock'),
 (4, 'Sharp Objects', 47.82, 'In stock'),
 (5, 'Sapiens: A Brief History of Humankind', 54.23, 'In stock')]

##### 10.3: Inserir um novo registro no banco

In [ ]:
cursor.execute("INSERT INTO livros (titulo, preco, estoque) VALUES ('Python for data analysis', 99.90, 'Out of stock');")
cursor.execute("INSERT INTO livros (titulo, preco, estoque) VALUES ('Python for data science', 89.90, 'Out of stock');")
cursor.execute("INSERT INTO livros (titulo, preco, estoque) VALUES ('Python for database', 79.90, 'Out of stock');")
cursor.execute("INSERT INTO livros (titulo, preco, estoque) VALUES ('Python for big data', 79.90, 'Out of stock');")
cursor.execute("INSERT INTO livros (titulo, preco, estoque) VALUES ('Python for intelligence and data analysis', 100, 'Out of stock');")

print("Registro armazenado no banco!")

Registro armazenado no banco!


##### Verifica quantos registros existem no banco

In [ ]:
cursor.execute("SELECT COUNT(*) FROM livros;")
cursor.fetchone()

(172,)

##### Verifica uma amostra dos últimos registros inseridos

In [ ]:
cursor.execute("SELECT * FROM livros ORDER BY id DESC;")
cursor.fetchmany(5)

[(173, 'Python for intelligence and data analysis', 100.0, 'Out of stock'),
 (172, 'Python for big data', 79.9, 'Out of stock'),
 (171, 'Python for database', 79.9, 'Out of stock'),
 (170, 'Python for data science', 89.9, 'Out of stock'),
 (169, 'Python for data analysis', 99.9, 'Out of stock')]

##### 10.4: `DELETE` (exclui registros)

In [ ]:
cursor.execute("DELETE FROM livros WHERE titulo = 'Python for data analysis';")
conn.commit()
print("Registro removido")

Registro removido


##### Verifica uma amostra dos últimos registros inseridos

In [ ]:
cursor.execute("SELECT * FROM livros ORDER BY id DESC;")
cursor.fetchmany(5)

[(173, 'Python for intelligence and data analysis', 100.0, 'Out of stock'),
 (172, 'Python for big data', 79.9, 'Out of stock'),
 (171, 'Python for database', 79.9, 'Out of stock'),
 (170, 'Python for data science', 89.9, 'Out of stock'),
 (168,
  'Rat Queens, Vol. 3: Demons (Rat Queens (Collected Editions) #11-15)',
  50.4,
  'In stock')]

#### Passo 11: DQL serve para **consultar dados** no banco:

- `SELECT`  
- `WHERE`  
- `ORDER BY`  
- `GROUP BY`  
- Funções de agregação (`COUNT`, `AVG`, `MAX`, `MIN`, `SUM`)

##### Consulta simples com `fetchone`

In [ ]:
cursor.execute("SELECT * FROM livros;")
cursor.fetchone()

(1, 'A Light in the Attic', 49.9, 'In stock')

##### Consulta simples com `fetchmany()`

In [ ]:
cursor.execute("SELECT * FROM livros;")
cursor.fetchmany(5)

[(1, 'A Light in the Attic', 49.9, 'In stock'),
 (2, 'Tipping the Velvet', 53.74, 'In stock'),
 (3, 'Soumission', 50.1, 'In stock'),
 (4, 'Sharp Objects', 47.82, 'In stock'),
 (5, 'Sapiens: A Brief History of Humankind', 54.23, 'In stock')]

##### Consulta simples com `fetchall()`

In [ ]:
cursor.execute("SELECT * FROM livros;")
cursor.fetchall()

[(1, 'A Light in the Attic', 49.9, 'In stock'),
 (2, 'Tipping the Velvet', 53.74, 'In stock'),
 (3, 'Soumission', 50.1, 'In stock'),
 (4, 'Sharp Objects', 47.82, 'In stock'),
 (5, 'Sapiens: A Brief History of Humankind', 54.23, 'In stock'),
 (6, 'The Requiem Red', 22.65, 'In stock'),
 (7, 'The Dirty Little Secrets of Getting Your Dream Job', 33.34, 'In stock'),
 (8,
  'The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull',
  17.93,
  'In stock'),
 (9,
  'The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics',
  22.6,
  'In stock'),
 (10, 'The Black Maria', 52.15, 'In stock'),
 (11, 'Starving Hearts (Triangular Trade Trilogy, #1)', 13.99, 'In stock'),
 (12, "Shakespeare's Sonnets", 20.66, 'In stock'),
 (13, 'Set Me Free', 17.46, 'In stock'),
 (14,
  "Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)",
  52.29,
  'In stock'),
 (15, 'Rip it Up and Start Again', 35.02, 'In stock'),
 (16,
  'Our Band Could Be 

#### `WHERE`
##### Consulta com filtro: Livros com preço abaixo de 20

In [ ]:
cursor.execute("SELECT titulo, preco FROM livros WHERE preco < 20;")
cursor.fetchall()

[('The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull',
  17.93),
 ('Starving Hearts (Triangular Trade Trilogy, #1)', 13.99),
 ('Set Me Free', 17.46),
 ('In Her Wake', 12.84),
 ('The Four Agreements: A Practical Guide to Personal Freedom', 17.66),
 ("Sophie's World", 15.94),
 ('Maude (1883-1993):She Grew Up with the country', 18.02),
 ('In a Dark, Dark Wood', 19.63),
 ('Untitled Collection: Sabbath Poems 2014', 14.27),
 ('Unicorn Tracks', 18.78),
 ('Tsubasa: WoRLD CHRoNiCLE 2 (Tsubasa WoRLD CHRoNiCLE #2)', 16.28),
 ('This One Summer', 19.49),
 ('Thirst', 17.27),
 ('The Torch Is Passed: A Harding Family Story', 19.09),
 ('The Life-Changing Magic of Tidying Up: The Japanese Art of Decluttering and Organizing',
  16.77),
 ('The Age of Genius: The Seventeenth Century and the Birth of the Modern Mind',
  19.73),
 ('Reskilling America: Learning to Labor in the Twenty-First Century', 19.83),
 ('The Coming Woman: A Novel Based on the Life of the Infamous Fe

#### `ORDER BY`
##### Consulta com ordenação: Top 5 livros mais caros

In [ ]:
cursor.execute("SELECT titulo, preco FROM livros ORDER BY preco DESC LIMIT 5;")
cursor.fetchall()

[('Python for AI', 150.9),
 ('Python for intelligence and data analysis', 100.0),
 ('L1', 99.99),
 ('L2', 99.99),
 ('L3', 99.99)]

#### `GROUP BY`
##### Consulta com agregação: Quantidade de livros por situação de estoque

In [ ]:
cursor.execute("SELECT estoque, COUNT(*) FROM livros GROUP BY estoque;")
cursor.fetchall()

[('In stock', 163), ('Out of stock', 7)]

#### `DISTINCT`
##### Consulta com filtro: Contagem de dados distintos

In [ ]:
cursor.execute("SELECT COUNT(DISTINCT preco) FROM livros;")
cursor.fetchall()

[(84,)]

#### `HAVING COUNT(*)`
##### Consulta com filtro: Contagem de dados repetidos

In [ ]:
cursor.execute("SELECT preco FROM livros GROUP BY preco HAVING COUNT(*)>1;")
cursor.fetchall()

[(12.84,),
 (13.99,),
 (14.27,),
 (15.94,),
 (16.28,),
 (16.77,),
 (17.27,),
 (17.46,),
 (17.66,),
 (17.93,),
 (18.02,),
 (18.78,),
 (19.09,),
 (19.49,),
 (19.63,),
 (19.73,),
 (19.83,),
 (20.59,),
 (20.66,),
 (22.5,),
 (22.6,),
 (22.65,),
 (23.11,),
 (23.82,),
 (23.88,),
 (23.89,),
 (25.02,),
 (25.27,),
 (25.52,),
 (26.41,),
 (30.52,),
 (31.05,),
 (31.12,),
 (31.77,),
 (32.24,),
 (33.29,),
 (33.34,),
 (33.63,),
 (34.53,),
 (35.02,),
 (36.89,),
 (37.13,),
 (37.32,),
 (37.59,),
 (38.16,),
 (39.25,),
 (39.58,),
 (40.3,),
 (40.76,),
 (41.83,),
 (42.96,),
 (44.18,),
 (45.07,),
 (45.17,),
 (45.22,),
 (47.61,),
 (47.82,),
 (49.05,),
 (49.9,),
 (50.1,),
 (50.27,),
 (50.4,),
 (51.04,),
 (51.33,),
 (52.15,),
 (52.22,),
 (52.29,),
 (53.13,),
 (53.74,),
 (54.11,),
 (54.23,),
 (54.64,),
 (56.06,),
 (56.13,),
 (56.41,),
 (56.5,),
 (57.25,),
 (57.31,),
 (58.11,),
 (79.9,),
 (89.9,),
 (99.99,)]

#### `AVG`
##### Consulta com agregação: Valor médio dos livros

In [ ]:
cursor.execute("SELECT AVG(preco) FROM livros ;")
cursor.fetchone()[0]

39.29394117647059

##### Consulta com agregação: Valor médio dos livros  (com arredondamento)

In [ ]:
cursor.execute("SELECT ROUND(AVG(preco), 2) FROM livros ;")
cursor.fetchone()[0]

39.29

#### `MAX`
##### Consulta com agregação: Maior valor entre os livros `MAX`

In [ ]:
cursor.execute("SELECT MAX(preco) FROM livros ;")
cursor.fetchone()[0]

150.9

#### `MIN`
##### Consulta com agregação: Menor valor entre os livros

In [ ]:
cursor.execute("SELECT MIN(preco) FROM livros ;")
cursor.fetchone()[0]

12.84

#### `SUM`
##### Consulta com agregação: Soma do valor dos livros

In [ ]:
cursor.execute("SELECT SUM(preco) FROM livros ;")
cursor.fetchone()[0]

6679.97

##### Consulta com agregação: Soma do valor dos livros em estoque

In [ ]:
cursor.execute("SELECT SUM(preco) FROM livros WHERE estoque = 'In stock';")
cursor.fetchone()[0]

6030.3

##### Consulta com agregação: Soma do valor dos livros sem estoque

In [ ]:
cursor.execute("SELECT SUM(preco) FROM livros WHERE estoque = 'Out of stock';")
cursor.fetchone()[0]

649.67

##### Consulta de multiplos valores: Valor Máximo, Medio e Mínimo

In [ ]:
# Se tiver múltiplas colunas, acesse pelo índice
cursor.execute("SELECT MAX(preco), AVG(preco), MIN(preco) FROM livros;")
maximo, medio, minimo = cursor.fetchone()
print(f'Preço Máximo: {maximo}')
print(f'Preço Médio: {medio}')
print(f'Preço Mínimo: {minimo}')

Preço Máximo: 150.9
Preço Médio: 39.29394117647059
Preço Mínimo: 12.84


##### Consulta com Ranking por Preço

In [ ]:
cursor.execute("SELECT RANK() OVER (ORDER BY preco DESC) as ranking, preco, titulo FROM livros;")
cursor.fetchall()
# pd.DataFrame(cursor.fetchall(), columns = ['Ranking','Preço','Título'])

[(1, 150.9, 'Python for AI'),
 (2, 100.0, 'Python for intelligence and data analysis'),
 (3, 99.99, 'L1'),
 (3, 99.99, 'L2'),
 (3, 99.99, 'L3'),
 (6, 89.9, 'Python for data science'),
 (6, 89.9, 'Python for data science'),
 (8, 79.9, 'Python for big data'),
 (8, 79.9, 'Python for database'),
 (8, 79.9, 'Python for big data'),
 (11, 58.11, 'The Death of Humanity: and the Case for Life'),
 (11, 58.11, 'The Death of Humanity: and the Case for Life'),
 (13, 57.31, 'Slow States of Collapse: Poems'),
 (13, 57.31, 'Slow States of Collapse: Poems'),
 (15,
  57.25,
  'Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991'),
 (15,
  57.25,
  'Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991'),
 (17, 56.5, 'The Past Never Ends'),
 (17, 56.5, 'The Past Never Ends'),
 (19,
  56.41,
  'The Pioneer Woman Cooks: Dinnertime: Comfort Classics, Freezer Food, 16-Minute Meals, and Other Delicious Ways to Solve Supper!'),
 (19,
  56.41,
  'The P

#### 12: JOINS – Relacionando tabelas

JOINS permitem **combinar dados de várias tabelas**.  
Vamos criar outras tabelas para exemplificar.

##### 12.1 Criar tabela de vendas

In [ ]:
cursor.execute('''
CREATE TABLE IF NOT EXISTS vendas (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    livro_id INTEGER,
    data_venda DATE,
    quantidade INTEGER,
    valor_total REAL,
    cliente TEXT,
    FOREIGN KEY (livro_id) REFERENCES livros(id)
);
''')
conn.commit()
print("Tabela criadas com sucesso!")

Tabela criadas com sucesso!


##### Verifica o esquema da tabela (estrutura das colunas)

In [ ]:
cursor.execute("PRAGMA table_info(vendas);")
for row in cursor.fetchall():
    print(row)

(0, 'id', 'INTEGER', 0, None, 1)
(1, 'livro_id', 'INTEGER', 0, None, 0)
(2, 'data_venda', 'DATE', 0, None, 0)
(3, 'quantidade', 'INTEGER', 0, None, 0)
(4, 'valor_total', 'REAL', 0, None, 0)
(5, 'cliente', 'TEXT', 0, None, 0)


##### 12.2: Inserir dados de exemplo

In [ ]:
vendas_exemplo = [
    (1, '2025-01-15', 2, 'João Silva'),
    (5, '2025-01-16', 1, 'Maria Santos'),
    (10, '2025-01-17', 3, 'Pedro Costa'),
    (15, '2025-01-18', 1, 'Ana Oliveira'),
    (20, '2025-01-19', 2, 'Carlos Lima'),
    (25, '2025-01-20', 1, 'Juliana Pereira'),
    (30, '2025-01-21', 2, 'Roberto Alves'),
    (35, '2025-01-22', 1, 'Fernanda Dias'),
    (40, '2025-01-23', 3, 'Ricardo Martins'),
    (45, '2025-01-24', 1, 'Patrícia Souza')
]

for livro_id, data, quantidade, cliente in vendas_exemplo:
    # Buscar preço
    cursor.execute("SELECT preco FROM livros WHERE id = ?", (livro_id,))
    resultado = cursor.fetchone()
    valor_total = round(resultado[0] * quantidade,2)

    cursor.execute('''
        INSERT INTO vendas (livro_id, data_venda, quantidade, valor_total, cliente)
        VALUES (?, ?, ?, ?, ?)
        ''', (livro_id, data, quantidade, valor_total, cliente))

conn.commit()
print(f"{len(vendas_exemplo)} vendas inseridas com sucesso!")

10 vendas inseridas com sucesso!


##### Verifica dados inseridos

In [ ]:
cursor.execute("SELECT * FROM vendas;")
cursor.fetchall()

[(1, 1, '2025-01-15', 2, 99.8, 'João Silva'),
 (2, 5, '2025-01-16', 1, 54.23, 'Maria Santos'),
 (3, 10, '2025-01-17', 3, 156.45, 'Pedro Costa'),
 (4, 15, '2025-01-18', 1, 35.02, 'Ana Oliveira'),
 (5, 20, '2025-01-19', 2, 90.34, 'Carlos Lima'),
 (6, 25, '2025-01-20', 1, 34.53, 'Juliana Pereira'),
 (7, 30, '2025-01-21', 2, 88.36, 'Roberto Alves'),
 (8, 35, '2025-01-22', 1, 15.94, 'Fernanda Dias'),
 (9, 40, '2025-01-23', 3, 100.89, 'Ricardo Martins'),
 (10, 45, '2025-01-24', 1, 45.07, 'Patrícia Souza'),
 (11, 1, '2025-01-15', 2, 99.8, 'João Silva'),
 (12, 5, '2025-01-16', 1, 54.23, 'Maria Santos'),
 (13, 10, '2025-01-17', 3, 156.45, 'Pedro Costa'),
 (14, 15, '2025-01-18', 1, 35.02, 'Ana Oliveira'),
 (15, 20, '2025-01-19', 2, 90.34, 'Carlos Lima'),
 (16, 25, '2025-01-20', 1, 34.53, 'Juliana Pereira'),
 (17, 30, '2025-01-21', 2, 88.36, 'Roberto Alves'),
 (18, 35, '2025-01-22', 1, 15.94, 'Fernanda Dias'),
 (19, 40, '2025-01-23', 3, 100.89, 'Ricardo Martins'),
 (20, 45, '2025-01-24', 1, 45.07

##### 12.3: Exemplos de JOINs com a tabela existente

In [ ]:
# 1. INNER JOIN - Livros que tiveram vendas
print("INNER JOIN - Livros Vendidos:")

cursor.execute('''
SELECT livros.titulo, livros.preco, vendas.quantidade, vendas.valor_total, vendas.data_venda
FROM livros
INNER JOIN vendas ON livros.rowid = vendas.livro_id
ORDER BY vendas.valor_total DESC;
''')

for row in cursor.fetchall():
    print(f"Título: {row[0]}")
    print(f"Preço: R$ {row[1]:.2f}")
    print(f"Venda: {row[2]} unid. - R$ {row[3]:.2f}")
    print(f"Data: {row[4]}")
    print()

INNER JOIN - Livros Vendidos:
Título: The Black Maria
Preço: R$ 52.15
Venda: 3 unid. - R$ 156.45
Data: 2025-01-17

Título: The Black Maria
Preço: R$ 52.15
Venda: 3 unid. - R$ 156.45
Data: 2025-01-17

Título: You can't bury them all: Poems
Preço: R$ 33.63
Venda: 3 unid. - R$ 100.89
Data: 2025-01-23

Título: You can't bury them all: Poems
Preço: R$ 33.63
Venda: 3 unid. - R$ 100.89
Data: 2025-01-23

Título: A Light in the Attic
Preço: R$ 49.90
Venda: 2 unid. - R$ 99.80
Data: 2025-01-15

Título: A Light in the Attic
Preço: R$ 49.90
Venda: 2 unid. - R$ 99.80
Data: 2025-01-15

Título: It's Only the Himalayas
Preço: R$ 45.17
Venda: 2 unid. - R$ 90.34
Data: 2025-01-19

Título: It's Only the Himalayas
Preço: R$ 45.17
Venda: 2 unid. - R$ 90.34
Data: 2025-01-19

Título: Wall and Piece
Preço: R$ 44.18
Venda: 2 unid. - R$ 88.36
Data: 2025-01-21

Título: Wall and Piece
Preço: R$ 44.18
Venda: 2 unid. - R$ 88.36
Data: 2025-01-21

Título: Sapiens: A Brief History of Humankind
Preço: R$ 54.23
Venda: 1 u

In [ ]:
# 2. LEFT JOIN - Todos os livros, mesmo os não vendidos
print("LEFT JOIN - Todos os Livros (com vendas):")

cursor.execute('''
SELECT
    livros.titulo,
    livros.preco,
    livros.estoque,
    COALESCE(vendas.quantidade, 0) as quantidade_vendida,
    COALESCE(vendas.valor_total, 0) as total_vendido
FROM livros
LEFT JOIN vendas ON livros.rowid = vendas.livro_id
ORDER BY total_vendido DESC;
''')
# COALESCE é uma função do SQL usada para substituir valores NULL por outro valor.

for row in cursor.fetchall():
    status_venda = "Vendido" if row[3] > 0 else " Não vendido"
    print(f"{row[0]:<80} | R$ {row[1]:6.2f} | {row[2]:<12} | {status_venda}")

LEFT JOIN - Todos os Livros (com vendas):
The Black Maria                                                                  | R$  52.15 | In stock     | Vendido
The Black Maria                                                                  | R$  52.15 | In stock     | Vendido
You can't bury them all: Poems                                                   | R$  33.63 | In stock     | Vendido
You can't bury them all: Poems                                                   | R$  33.63 | In stock     | Vendido
A Light in the Attic                                                             | R$  49.90 | In stock     | Vendido
A Light in the Attic                                                             | R$  49.90 | In stock     | Vendido
It's Only the Himalayas                                                          | R$  45.17 | In stock     | Vendido
It's Only the Himalayas                                                          | R$  45.17 | In stock     | Vendido
Wall and Piece

In [ ]:
# 3. JOIN com agregação - Total de vendas por livro
print("JOIN com Agregação - Resumo de Vendas:")

cursor.execute('''
SELECT
    livros.titulo,
    livros.preco,
    COUNT(vendas.id) as total_vendas,
    SUM(vendas.quantidade) as total_unidades,
    SUM(vendas.valor_total) as valor_total
FROM livros
LEFT JOIN vendas ON livros.rowid = vendas.livro_id
GROUP BY livros.rowid
ORDER BY valor_total DESC NULLS LAST;
''')

for row in cursor.fetchall():
    print(f"Título: {row[0]:<40}")
    print(f"Preço: R$ {row[1]:6.2f}")
    print(f" Vendas: {row[2]} transações")
    print(f"Unidades: {row[3] or 0}")
    print(f"Total: R$ {row[4] or 0:.2f}")
    print()

JOIN com Agregação - Resumo de Vendas:
Título: The Black Maria                         
Preço: R$  52.15
 Vendas: 2 transações
Unidades: 6
Total: R$ 312.90

Título: You can't bury them all: Poems          
Preço: R$  33.63
 Vendas: 2 transações
Unidades: 6
Total: R$ 201.78

Título: A Light in the Attic                    
Preço: R$  49.90
 Vendas: 2 transações
Unidades: 4
Total: R$ 199.60

Título: It's Only the Himalayas                 
Preço: R$  45.17
 Vendas: 2 transações
Unidades: 4
Total: R$ 180.68

Título: Wall and Piece                          
Preço: R$  44.18
 Vendas: 2 transações
Unidades: 4
Total: R$ 176.72

Título: Sapiens: A Brief History of Humankind   
Preço: R$  54.23
 Vendas: 2 transações
Unidades: 2
Total: R$ 108.46

Título: Without Borders (Wanderlove #1)         
Preço: R$  45.07
 Vendas: 2 transações
Unidades: 2
Total: R$ 90.14

Título: Rip it Up and Start Again               
Preço: R$  35.02
 Vendas: 2 transações
Unidades: 2
Total: R$ 70.04

Título: Black Dust 

In [ ]:
# 4. JOIN com filtro WHERE
print("JOIN com Filtro - Livros em Estoque com Vendas:")

cursor.execute('''
SELECT
    livros.titulo,
    livros.estoque,
    vendas.quantidade,
    vendas.data_venda,
    vendas.cliente
FROM livros
INNER JOIN vendas ON livros.rowid = vendas.livro_id
WHERE livros.estoque = 'In stock'
ORDER BY vendas.data_venda DESC;
''')

for row in cursor.fetchall():
    print(f"Título: {row[0]}")
    print(f"Estoque: {row[1]}")
    print(f"Quantidade: {row[2]}")
    print(f"Data: {row[3]}")
    print(f"Cliente: {row[4]}")
    print()

JOIN com Filtro - Livros em Estoque com Vendas:
Título: Without Borders (Wanderlove #1)
Estoque: In stock
Quantidade: 1
Data: 2025-01-24
Cliente: Patrícia Souza

Título: Without Borders (Wanderlove #1)
Estoque: In stock
Quantidade: 1
Data: 2025-01-24
Cliente: Patrícia Souza

Título: You can't bury them all: Poems
Estoque: In stock
Quantidade: 3
Data: 2025-01-23
Cliente: Ricardo Martins

Título: You can't bury them all: Poems
Estoque: In stock
Quantidade: 3
Data: 2025-01-23
Cliente: Ricardo Martins

Título: Sophie's World
Estoque: In stock
Quantidade: 1
Data: 2025-01-22
Cliente: Fernanda Dias

Título: Sophie's World
Estoque: In stock
Quantidade: 1
Data: 2025-01-22
Cliente: Fernanda Dias

Título: Wall and Piece
Estoque: In stock
Quantidade: 2
Data: 2025-01-21
Cliente: Roberto Alves

Título: Wall and Piece
Estoque: In stock
Quantidade: 2
Data: 2025-01-21
Cliente: Roberto Alves

Título: Black Dust
Estoque: In stock
Quantidade: 1
Data: 2025-01-20
Cliente: Juliana Pereira

Título: Black Dust

In [ ]:
# 5. Múltiplos JOINS (se tivesse outras tabelas)
print("Estatísticas Gerais:")

# Total de livros
cursor.execute("SELECT COUNT(*) FROM livros")
total_livros = cursor.fetchone()[0]

# Livros com vendas
cursor.execute("SELECT COUNT(DISTINCT livro_id) FROM vendas")
livros_com_vendas = cursor.fetchone()[0]

# Total arrecadado
cursor.execute("SELECT SUM(valor_total) FROM vendas")
total_arrecadado = cursor.fetchone()[0] or 0

print(f"Total de livros: {total_livros}")
print(f"Livros com vendas: {livros_com_vendas}")
print(f"Total arrecadado: R$ {total_arrecadado:.2f}")
print(f"Percentual vendido: {(livros_com_vendas/total_livros)*100:.1f}%")

Estatísticas Gerais:
Total de livros: 170
Livros com vendas: 10
Total arrecadado: R$ 1441.26
Percentual vendido: 5.9%


#### 13: Transações – BEGIN, COMMIT e ROLLBACK

Uma transação é um conjunto de operações que devem ser executadas juntas.  

- `BEGIN` → inicia a transação  
- `COMMIT` → confirma as alterações  
- `ROLLBACK` → desfaz as alterações (caso de erro)  

In [ ]:
# Exemplo de transação com ROLLBACK
try:
    conn.execute("BEGIN;") # Inicia a trasação
    cursor.execute("INSERT INTO livros (titulo, preco, estoque) VALUES ('´Super Python', 99.90, 'Out of stock');") # Cria novo registro
    raise Exception("Erro simulado!")  # Força erro
    conn.commit() # Se chegasse aqui, faria commit
except:
    conn.rollback() # Reverte em caso de erro
    print("Transação revertida por erro")

Transação revertida por erro


#### Passo 13: Encerra conexão com banco de dados

In [ ]:
cursor.close()
conn.close()
print("Conexão encerrada")

Conexão encerrada
